# ATDL Assignment 1 — Task 2: Ternary Quantizer & Student Architecture

**Goal (per the assignment):** design a ResNet18 student whose conv/FC weights are constrained to the ternary set {−α, 0, +α}, where α is a learnable/derived per-layer or per-channel scale.

**No training happens in this notebook.** This is pure architecture: the quantizer function, the STE that lets gradients flow through it, the ternary-aware layers, and the full `TernaryResNet18`. Everything here runs in seconds on CPU — you don't need a GPU for this notebook.

**Required reading this ties to:**
- Li, Zhang, Liu, *Ternary Weight Networks* (TWN, 2016) — the Δ/α formula implemented below.
- Bengio, Léonard, Courville, *Estimating or Propagating Gradients Through Stochastic Neurons* — the Straight-Through Estimator, implemented as a custom `torch.autograd.Function`.

**What this notebook produces, in order:**
1. The core `ternary_quantize` function (the math).
2. The STE wrapper that makes it usable inside backprop.
3. A quick self-test proving gradients actually reach the latent weight.
4. `TernaryConv2d` / `TernaryLinear` — drop-in replacements for `nn.Conv2d`/`nn.Linear`.
5. `TernaryBasicBlock` / `TernaryResNetCIFAR` / `TernaryResNet18` — the full student, with the first conv and final FC deliberately kept FP32 (documented below, not just implicit in the code).
6. `verify_ternary_constraint` — the checkpoint-verification script required as a deliverable.
7. A compatibility check against your Task 4 baseline checkpoint, since Task 3 will warm-start the student's latent weights from it.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_DIR = "/content/drive/MyDrive/atdl"
    os.makedirs(DRIVE_DIR, exist_ok=True)
else:
    DRIVE_DIR = "."

BASELINE_CKPT = os.path.join(DRIVE_DIR, "resnet18_cifar10_fp32_baseline_best.pth")
print(f"Looking for Task 4 baseline checkpoint at: {BASELINE_CKPT}")
print("Found." if os.path.exists(BASELINE_CKPT) else "Not found yet -- that's fine, only the last cell of this notebook needs it.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Looking for Task 4 baseline checkpoint at: /content/drive/MyDrive/atdl/resnet18_cifar10_fp32_baseline_best.pth
Found.


In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")  # no GPU needed for this notebook
print("This notebook runs on CPU -- no training here, just architecture + quick tests.")

This notebook runs on CPU -- no training here, just architecture + quick tests.


## The ternary quantization formula (TWN)

For a weight tensor `W` (real-valued, FP32), the ternary version is computed as:

```
delta   = 0.7 * mean(|W|)                       # threshold
W_q[i]  =  +alpha   if W[i] >  delta
        =  -alpha   if W[i] < -delta
        =   0       otherwise
alpha   = mean(|W[i]|)  over weights where |W[i]| > delta
```

Two modes are supported:
- **`per_channel=True`** (used for the student): delta/alpha computed *separately for each output channel* — a per-output-filter scale, which is finer-grained than one global scale for the whole tensor and generally preserves more accuracy.
- **`per_channel=False`**: one delta/alpha for the entire tensor — the simpler per-tensor variant, worth including in your report as the baseline TWN doesn't strictly require per-channel granularity.

Note `alpha` is **derived**, not learned by gradient descent, in this TWN-style formula — it falls out of whatever the latent weights currently are. (The TTQ paper by Zhu et al. instead makes alpha a learnable scalar per layer via its own gradient; worth a sentence in your report contrasting the two, even though this notebook implements TWN.)

In [ ]:
def ternary_quantize(w, per_channel=True, delta_factor=0.7):
    """Returns (w_q, alpha, delta). w_q has the same shape as w.
    alpha/delta are per-output-channel if per_channel=True, else scalars."""
    orig_shape = w.shape
    if per_channel:
        w_flat = w.reshape(w.size(0), -1)
        delta = delta_factor * w_flat.abs().mean(dim=1, keepdim=True)
        mask = (w_flat.abs() > delta).float()
        denom = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        alpha = (w_flat.abs() * mask).sum(dim=1, keepdim=True) / denom
        w_q = (alpha * mask * torch.sign(w_flat)).reshape(orig_shape)
        return w_q, alpha.view(-1), delta.view(-1)
    else:
        delta = delta_factor * w.abs().mean()
        mask = (w.abs() > delta).float()
        denom = mask.sum().clamp(min=1.0)
        alpha = (w.abs() * mask).sum() / denom
        w_q = alpha * mask * torch.sign(w)
        return w_q, alpha, delta

## Quick sanity check on the quantizer alone

Before wiring this into a layer, confirm on a random tensor that: (a) each output channel really does collapse to exactly 3 unique values (or fewer, if a channel happens to be all-zero after thresholding), and (b) alpha is a sensible positive number close to the magnitude of the "kept" weights.

In [ ]:
test_w = torch.randn(8, 16, 3, 3)  # pretend conv weight: 8 out-channels, 16 in, 3x3
w_q, alpha, delta = ternary_quantize(test_w, per_channel=True)

print("alpha per channel:", alpha.detach().numpy().round(4))
print("delta per channel:", delta.detach().numpy().round(4))

for c in range(test_w.size(0)):
    unique_vals = torch.unique(w_q[c]).numel()
    assert unique_vals <= 3, f"Channel {c} has {unique_vals} unique values, expected <= 3"
sparsity = (w_q == 0).float().mean().item()
print(f"All {test_w.size(0)} channels have <= 3 unique values. Overall sparsity: {sparsity:.3f}")

alpha per channel: [1.1308 1.2717 1.0509 1.1842 1.2342 1.243  1.2815 1.1414]
delta per channel: [0.561  0.5735 0.4975 0.5773 0.5533 0.5794 0.5979 0.5159]
All 8 channels have <= 3 unique values. Overall sparsity: 0.430


## The Straight-Through Estimator (STE)

`ternary_quantize` involves a hard threshold and `sign()` — both have zero gradient almost everywhere, so backprop through them directly would kill the gradient signal to the latent weights. The STE sidesteps this with a custom `torch.autograd.Function`:

- **Forward**: compute and return the real ternary-quantized tensor. The network genuinely computes with only 3 values.
- **Backward**: pretend the quantizer was the identity function, so the incoming gradient is passed straight through unchanged to the latent weight (this is exactly the mechanism analyzed in the Bengio et al. and Yin et al. papers).

An optional `clip_grad` mode is included: if enabled, gradients are zeroed for latent weights whose magnitude already exceeds `clip_value`, which is a common STE refinement to prevent runaway latent-weight growth. It's off by default here (`clip_grad=False`) — worth trying as an extra ablation if you have time, and worth mentioning as a known variant either way.

In [ ]:
class _TernarySTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, w, delta_factor, per_channel, clip_grad, clip_value):
        w_q, _, _ = ternary_quantize(w, per_channel=per_channel, delta_factor=delta_factor)
        ctx.clip_grad = clip_grad
        if clip_grad:
            ctx.save_for_backward(w)
            ctx.clip_value = clip_value
        return w_q

    @staticmethod
    def backward(ctx, grad_output):
        if ctx.clip_grad:
            (w,) = ctx.saved_tensors
            grad_input = grad_output.clone()
            grad_input[w.abs() > ctx.clip_value] = 0.0
        else:
            grad_input = grad_output
        return grad_input, None, None, None, None


def ternary_quantize_ste(w, delta_factor=0.7, per_channel=True, clip_grad=False, clip_value=1.0):
    return _TernarySTE.apply(w, delta_factor, per_channel, clip_grad, clip_value)

## Sanity check: does the gradient actually reach the latent weight?

This is the single most important thing to verify in this whole notebook. If this cell failed (`grad` were `None` or all zeros), the STE would be broken and no amount of training in Task 3 would actually update the ternary layers' latent weights.

In [ ]:
latent_w = torch.randn(4, 4, 3, 3, requires_grad=True)
w_q = ternary_quantize_ste(latent_w, delta_factor=0.7, per_channel=True)

fake_loss = w_q.sum()
fake_loss.backward()

assert latent_w.grad is not None, "STE broke -- no gradient reached the latent weight!"
assert latent_w.grad.abs().sum().item() > 0, "Gradient reached but is all zero -- STE broke!"
print("Gradient successfully passed through the STE to the latent FP32 weight.")
print(f"Gradient shape: {latent_w.grad.shape}, nonzero entries: {(latent_w.grad != 0).sum().item()}/{latent_w.grad.numel()}")

Gradient successfully passed through the STE to the latent FP32 weight.
Gradient shape: torch.Size([4, 4, 3, 3]), nonzero entries: 144/144


## Ternary-aware layers: `TernaryConv2d` / `TernaryLinear`

Both hold `self.weight` as a normal `nn.Parameter` — this **is** the latent FP32 weight, the only thing the optimizer ever touches directly. On every `forward()` call, `ternary_quantize_ste` produces a ternary version on the fly and that's what actually convolves/multiplies with the input. The latent weight is never itself overwritten with ternary values — quantization is a forward-pass-only operation, exactly as the rubric requires ("latent FP32 weights maintained separately from quantized forward weights").

In [ ]:
class TernaryConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0,
                 bias=False, delta_factor=0.7, per_channel=True,
                 clip_grad=False, clip_value=1.0):
        super().__init__()
        self.weight = nn.Parameter(
            torch.empty(out_channels, in_channels, kernel_size, kernel_size))
        nn.init.kaiming_normal_(self.weight, mode="fan_out", nonlinearity="relu")
        self.bias = nn.Parameter(torch.zeros(out_channels)) if bias else None
        self.stride = stride
        self.padding = padding
        self.delta_factor = delta_factor
        self.per_channel = per_channel
        self.clip_grad = clip_grad
        self.clip_value = clip_value

    def forward(self, x):
        w_q = ternary_quantize_ste(self.weight, self.delta_factor, self.per_channel,
                                    self.clip_grad, self.clip_value)
        return F.conv2d(x, w_q, self.bias, stride=self.stride, padding=self.padding)


class TernaryLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True, delta_factor=0.7,
                 per_channel=True, clip_grad=False, clip_value=1.0):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.kaiming_uniform_(self.weight, a=5 ** 0.5)
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        self.delta_factor = delta_factor
        self.per_channel = per_channel
        self.clip_grad = clip_grad
        self.clip_value = clip_value

    def forward(self, x):
        w_q = ternary_quantize_ste(self.weight, self.delta_factor, self.per_channel,
                                    self.clip_grad, self.clip_value)
        return F.linear(x, w_q, self.bias)

## `TernaryResNet18` — the full student

Architecture choice, documented explicitly here (this is the sentence the rubric wants in your report, not just implicit in code): **the first conv layer and the final FC layer are kept FP32** (plain `nn.Conv2d` / `nn.Linear`, not their Ternary counterparts). Two reasons:
1. The first layer reads raw RGB pixels directly — ternarizing it discards fine-grained color/intensity information right at the input, before the network has had any chance to build up robustness to that loss.
2. Both layers are tiny relative to the rest of the network (the stem conv and the 512→10 FC layer together are a small fraction of total parameters), so ternarizing them buys almost no extra compression while risking a disproportionate accuracy hit.

`ternary_fc=False` by default reflects this; it's exposed as a flag so you can test ternarizing the FC layer too as an extra ablation if you want a "how far can we push it" data point for your discussion section.

Every other conv in every `TernaryBasicBlock` (including the 1x1 projection convs in the shortcut paths) is ternary.

In [ ]:
class TernaryBasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, ternary_kwargs=None):
        super().__init__()
        tk = ternary_kwargs or {}
        self.conv1 = TernaryConv2d(in_planes, planes, kernel_size=3, stride=stride,
                                    padding=1, bias=False, **tk)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = TernaryConv2d(planes, planes, kernel_size=3, stride=1,
                                    padding=1, bias=False, **tk)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                TernaryConv2d(in_planes, self.expansion * planes, kernel_size=1,
                              stride=stride, bias=False, **tk),
                nn.BatchNorm2d(self.expansion * planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out)


class TernaryResNetCIFAR(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10, ternary_kwargs=None,
                 ternary_fc=False):
        super().__init__()
        self.in_planes = 64
        # First conv stays FP32 -- see markdown above for why.
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], 1, ternary_kwargs)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], 2, ternary_kwargs)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], 2, ternary_kwargs)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], 2, ternary_kwargs)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        # Final FC stays FP32 by default -- see markdown above for why.
        if ternary_fc:
            self.fc = TernaryLinear(512 * block.expansion, num_classes,
                                     **(ternary_kwargs or {}))
        else:
            self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride, ternary_kwargs):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s, ternary_kwargs=ternary_kwargs))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        return self.fc(out)


def TernaryResNet18(num_classes=10, delta_factor=0.7, per_channel=True,
                     clip_grad=False, clip_value=1.0, ternary_fc=False):
    ternary_kwargs = dict(delta_factor=delta_factor, per_channel=per_channel,
                           clip_grad=clip_grad, clip_value=clip_value)
    return TernaryResNetCIFAR(TernaryBasicBlock, [2, 2, 2, 2], num_classes,
                               ternary_kwargs=ternary_kwargs, ternary_fc=ternary_fc)

## End-to-end self-test: build, forward, backward on a dummy batch

This confirms the whole model (not just isolated layers) is wired correctly before Task 3 spends real training time on it: shapes are right, a forward pass runs, and gradients reach every ternary layer's latent weight.

In [ ]:
student = TernaryResNet18(num_classes=10, delta_factor=0.7, per_channel=True, ternary_fc=False)

n_ternary_conv = sum(1 for m in student.modules() if isinstance(m, TernaryConv2d))
n_ternary_linear = sum(1 for m in student.modules() if isinstance(m, TernaryLinear))
n_total_params = sum(p.numel() for p in student.parameters())
n_fp32_kept_params = student.conv1.weight.numel() + student.fc.weight.numel() + student.fc.bias.numel()

print(f"TernaryConv2d layers: {n_ternary_conv}")
print(f"TernaryLinear layers: {n_ternary_linear} (0 expected, since ternary_fc=False)")
print(f"Total parameters: {n_total_params:,}")
print(f"Parameters kept FP32 (stem conv + FC): {n_fp32_kept_params:,} "
      f"({100 * n_fp32_kept_params / n_total_params:.2f}% of total)")

dummy_x = torch.randn(4, 3, 32, 32)   # batch of 4 CIFAR-sized images
dummy_y = torch.randint(0, 10, (4,))

out = student(dummy_x)
assert out.shape == (4, 10), f"Expected (4, 10), got {out.shape}"
print(f"\nForward pass OK, output shape: {out.shape}")

loss = F.cross_entropy(out, dummy_y)
loss.backward()

n_with_grad = sum(1 for p in student.parameters() if p.grad is not None and p.grad.abs().sum() > 0)
n_params_total = sum(1 for _ in student.parameters())
print(f"Backward pass OK. Parameters with nonzero gradient: {n_with_grad}/{n_params_total}")
assert n_with_grad == n_params_total, "Some parameters got no gradient -- check the architecture!"
print("Every parameter (including every ternary layer's latent weight) received a gradient.")

TernaryConv2d layers: 19
TernaryLinear layers: 0 (0 expected, since ternary_fc=False)
Total parameters: 11,173,962
Parameters kept FP32 (stem conv + FC): 6,858 (0.06% of total)

Forward pass OK, output shape: torch.Size([4, 10])
Backward pass OK. Parameters with nonzero gradient: 62/62
Every parameter (including every ternary layer's latent weight) received a gradient.


## Checkpoint verification (Deliverable #3 requirement)

The assignment requires "the script used to load and verify the ternary constraint on the saved weights." `verify_ternary_constraint` does this: for every ternary layer, it re-derives what the ternary version *should* look like from the current latent weights, and checks the two match (within floating-point tolerance) and reports per-layer sparsity. This is the script you'll point to for that deliverable, and it's also what Task 3 will call at the end of training to confirm the final saved checkpoint genuinely respects the ternary constraint.

In [ ]:
def verify_ternary_constraint(model, tol=1e-5):
    results = {}
    all_ok = True
    for name, module in model.named_modules():
        if isinstance(module, (TernaryConv2d, TernaryLinear)):
            w = module.weight.detach()
            w_q, alpha, delta = ternary_quantize(w, per_channel=module.per_channel,
                                                  delta_factor=module.delta_factor)
            if module.per_channel:
                w_q_flat = w_q.reshape(w_q.size(0), -1)
                ok = True
                for c in range(w_q_flat.size(0)):
                    vals = torch.unique(w_q_flat[c])
                    for v in vals.tolist():
                        if abs(v) > tol and abs(abs(v) - alpha[c].item()) > tol:
                            ok = False
            else:
                vals = torch.unique(w_q)
                ok = all(abs(v.item()) < tol or abs(abs(v.item()) - alpha.item()) < tol
                          for v in vals)
            sparsity = (w_q == 0).float().mean().item()
            results[name] = {
                "ok": ok,
                "sparsity": round(sparsity, 4),
                "alpha_mean": round(alpha.mean().item(), 6),
                "num_unique_values": torch.unique(w_q).numel(),
            }
            all_ok = all_ok and ok
    return results, all_ok


results, all_ok = verify_ternary_constraint(student)
n_layers = len(results)
n_pass = sum(r["ok"] for r in results.values())
avg_sparsity = sum(r["sparsity"] for r in results.values()) / n_layers
print(f"Ternary constraint check: {n_pass}/{n_layers} layers OK (all_ok={all_ok})")
print(f"Average per-layer sparsity: {avg_sparsity:.3f}")
print(f"(Randomly initialized weights here -- sparsity/alpha will look different once "
      f"trained in Task 3. This is just confirming the check itself works.)")

Ternary constraint check: 19/19 layers OK (all_ok=True)
Average per-layer sparsity: 0.424
(Randomly initialized weights here -- sparsity/alpha will look different once trained in Task 3. This is just confirming the check itself works.)


## Loading FP32 weights into the student (needed for Task 3's warm start)

`load_fp32_init` copies an FP32 checkpoint's weights directly into the ternary student's *latent* parameters -- this works because `TernaryConv2d`/`TernaryLinear` name their weight `self.weight`, matching `nn.Conv2d`/`nn.Linear`'s naming exactly, so `state_dict()` keys line up one-to-one. The values loaded in are still plain FP32 numbers; they just get ternarized on every forward pass from that point on. If your Task 4 baseline checkpoint already exists in `atdl/`, this cell will actually test the load; otherwise it just defines the function for later.

In [ ]:
def load_fp32_init(ternary_model, fp32_state_dict, strict=True):
    missing, unexpected = ternary_model.load_state_dict(fp32_state_dict, strict=False)
    if strict and (missing or unexpected):
        raise RuntimeError(
            f"Checkpoint does not match student architecture.\n"
            f"Missing keys: {missing}\nUnexpected keys: {unexpected}"
        )
    return missing, unexpected


if os.path.exists(BASELINE_CKPT):
    baseline_ckpt = torch.load(BASELINE_CKPT, map_location="cpu")
    fresh_student = TernaryResNet18(num_classes=10, delta_factor=0.7, per_channel=True, ternary_fc=False)
    missing, unexpected = load_fp32_init(fresh_student, baseline_ckpt["model_state_dict"], strict=True)
    print(f"Loaded Task 4 baseline checkpoint into a fresh ternary student successfully.")
    print(f"Missing keys: {len(missing)}, unexpected keys: {len(unexpected)} (both should be 0)")
    results, all_ok = verify_ternary_constraint(fresh_student)
    print(f"Ternary constraint after warm-start load: {sum(r['ok'] for r in results.values())}/{len(results)} "
          f"layers OK -- this is expected to already pass, since quantization is applied fresh on every "
          f"forward pass regardless of what the latent weights currently are.")
else:
    print(f"Baseline checkpoint not found at {BASELINE_CKPT} yet -- once Task 4 finishes training, "
          f"re-run this cell to confirm the warm-start load works before Task 3.")

Loaded Task 4 baseline checkpoint into a fresh ternary student successfully.
Missing keys: 0, unexpected keys: 0 (both should be 0)
Ternary constraint after warm-start load: 19/19 layers OK -- this is expected to already pass, since quantization is applied fresh on every forward pass regardless of what the latent weights currently are.
